# Mini-LLM (decoder-only Transformer) — Colab training

1. GPU check → 2. install deps → 3. data → 4. tokenizer → 5. model → 6. train → 7. eval + samples → 8. download artifacts.
Preset order: run `tiny` first; scale to `small`/`medium` after it works.

In [ ]:
# 1. GPU check
!nvidia-smi
import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print('bf16 supported:', torch.cuda.is_bf16_supported())

In [ ]:
# 2b. (strongly recommended) Back up checkpoints to Google Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/mini-llm && ln -sfn /content/drive/MyDrive/mini-llm /content/drive_out
!cp -r checkpoints /content/drive/MyDrive/mini-llm/ 2>/dev/null; echo backup-done
# run this cell again any time to re-sync checkpoints to Drive

In [ ]:
# 2. Get project code (upload this repo/folder to Colab first, or clone your git URL)
# Option A: upload LyraAi folder to /content and cd into it
# Option B: clone:
# !git clone <YOUR_REPO_URL> && %cd <REPO>
%cd /content/LyraAi
!pip install -q -r requirements.txt
!python -c "import torch, tokenizers; print('deps ok')"

In [ ]:
# 3. Prepare dataset (generates data/conversations.jsonl + pretrain_corpus.txt)
!python scripts/build_data.py
!wc -l data/conversations.jsonl && head -c 600 data/conversations.jsonl

In [ ]:
# 4. Train tokenizer (tiny preset: vocab 8000)
!python -m tokenizer.train_tokenizer --config config/tiny.json

In [ ]:
# 5. Sanity: param count + forward + loss (CPU/GPU, seconds)
!python - <<'EOF'
from model.config import ModelConfig
from model.model import MiniGPT
import torch
import json
cfg=json.load(open('config/tiny.json'))
mc=cfg['model']
c=ModelConfig(**{k:v for k,v in mc.items() if k in ModelConfig.__dataclass_fields__})
m=MiniGPT(c)
print(f"params: {m.count_params()/1e6:.2f}M")
x=torch.randint(0,c.vocab_size,(2,32))
out=m(x, labels=x)
print('logits', tuple(out['logits'].shape), 'loss', float(out['loss']))
EOF

In [ ]:
# 6a. (optional but recommended) Stage-1 pretraining: plain language modeling
!python -m training.train --config config/tiny.json --stage pretrain

In [ ]:
# 6b. Stage-2 SFT: conversation fine-tuning (assistant-only loss)
# To start from pretrained weights, set train.resume_from to the pretrain export dir first.
!python -m training.train --config config/tiny.json --stage sft

In [ ]:
# 7. Quality gate: RU/EN prompt set
!python scripts/evaluate.py --checkpoint checkpoints/tiny_run/export

In [ ]:
# 8. Prove it's a real autoregressive LLM (8 checks)
!python -m tests.test_generation --checkpoint checkpoints/tiny_run/export

In [ ]:
# 9. Resume training (after reconnect: restore checkpoints from Drive first)
# !cp -r /content/drive/MyDrive/mini-llm/checkpoints ./ 2>/dev/null; ls checkpoints/
!python -m training.train --config config/tiny.json --stage sft --resume checkpoints/tiny_run/last

In [ ]:
# 10. Pack + download artifacts (model + tokenizer, no retraining needed for chat)
!ls -lh checkpoints/tiny_run/export/ checkpoints/tiny_run/*.pt
from google.colab import files
!zip -r mini-llm-tiny.zip checkpoints/tiny_run/export checkpoints/tiny_run/tokenizer.json config/tiny.json
files.download('mini-llm-tiny.zip')